# 00_config — 공통 설정 로더 (담당: core)

- **실험 숫자·파라미터는 [`contract/params.json`](../contract/params.json)에만 둡니다.** 숫자나 하이퍼파라미터(`XGB_PARAMS`, `TUNE`, `DELTA`, `B_TEST`, `STAGE` 등)를 바꿀 때는 이 노트북을 고치지 않고 `contract/params.json`만 수정하면 됩니다.
- 미결 항목을 확정할 때는 `contract/params.json`에서 값을 채우고 `"PENDING"`에서 해당 키를 지웁니다.
- 모든 노트북은 `notebooks/` 안에서 기존과 똑같이 `%run -i ./00_config.ipynb`로 불러 씁니다(팀원 노트북 수정 불필요).
- **범위별 해시 (`config_fingerprint(scope)`):**
  - 기본 `config_fingerprint()`(= `scope="shared"`)는 Dev OOB 공통 비교 키(`SEED`, `SAMPLE`, `PREP`, `B`, `MIN_RATE`, `SR_MODE`, `RATES`)만 해시합니다.
  - 따라서 `XGB_PARAMS`, `DELTA`, `B_TEST`, `EXPLAIN`, `ROBUST`, `STAGE` 등이 바뀌어도 영향받지 않는 기존 모형 결과가 막히지 않습니다.

In [ ]:
# 1. 경로: 내려받은 프로젝트 폴더 기준
# 보통은 None으로 두면 자동으로 찾습니다. 자동으로 못 찾을 때만 core 담당이 경로를 넣습니다(팀원은 고치지 않음).
PROJECT_DIR = None   # 예: r"C:/Users/이름/Downloads/lendingclub"  또는  "/Users/이름/Downloads/lendingclub"

import os, sys, json, hashlib, platform, subprocess
from importlib import metadata
from pathlib import Path
import numpy as np

def find_root():
    """PROJECT_DIR → LC_ROOT 환경변수 → 현재 폴더에서 위로 올라가며 notebooks/00_config.ipynb가 있는 폴더."""
    for given in (PROJECT_DIR, os.environ.get("LC_ROOT")):
        if given:
            p = Path(given).expanduser().resolve()
            assert (p / "notebooks" / "00_config.ipynb").exists(), f"이 경로에 notebooks/00_config.ipynb가 없음: {p}"
            return p
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "notebooks" / "00_config.ipynb").exists():
            return d
    raise FileNotFoundError("프로젝트 폴더를 못 찾음. notebooks/ 안에서 실행하세요.")

ROOT = find_root()
RAW_DIR, NB_DIR, DATA_DIR, CONTRACT_DIR, OUT_DIR, REPORT_DIR = (
    ROOT / d for d in ("raw", "notebooks", "data", "contract", "outputs", "report"))
FREEZE_DIR   = ROOT / "freeze"
PARAMS_JSON  = Path(os.environ["LC_PARAMS_JSON"]).expanduser().resolve() if os.environ.get("LC_PARAMS_JSON") else (CONTRACT_DIR / "params.json")
TRAIN_CSV    = RAW_DIR / "lending_club_2020_train_new.csv"  # 01_prep(core 담당)만 필요
TREASURY_CSV = RAW_DIR / "us_treasury_3y_5y_monthly.csv"
# Test 파일 경로는 두지 않는다. 사전 고정 뒤 00b_test_prep·05_test에서만 지정한다.

In [ ]:
# 2. 실험 설정 로드 (contract/params.json에서 숫자·파라미터를 읽어 기존 전역 변수로 노출)
assert PARAMS_JSON.exists(), f"설정 파일 없음: {PARAMS_JSON}"
_P = json.loads(PARAMS_JSON.read_text(encoding="utf-8"))

PENDING = dict(_P.get("PENDING") or {})
def pending(name, dev_value, note):
    """하위 호환용 헬퍼: PENDING에 항목을 남기고 dev_value를 돌려준다."""
    PENDING[name] = note
    return dev_value

CONFIG_VERSION = _P["CONFIG_VERSION"]
STAGE          = _P["STAGE"]              # "dev" | "final"
SEED           = int(_P["SEED"])

SAMPLE         = dict(_P["SAMPLE"])
EXPECT         = dict(_P["EXPECT"])
PREP           = dict(_P["PREP"])

# 표시용 항목: 실제 값은 lib_prep에 있음 (해시에 영향 없음)
SIGN_TABLE     = _P.get("SIGN_TABLE", "")
RECOVERY_LAG   = _P.get("RECOVERY_LAG", "")
SENS_SETS      = _P.get("SENS_SETS", "")

B              = int(_P["B"])
B_TEST         = int(_P["B_TEST"])
_r             = _P["RATES"]
RATES          = np.round(np.linspace(float(_r[0]), float(_r[1]), int(_r[2])), 2) if isinstance(_r, list) and len(_r) == 3 else np.asarray(_r, dtype=float)
MIN_RATE       = float(_P["MIN_RATE"])
SR_MODE        = _P["SR_MODE"]
REF_MODEL      = _P["REF_MODEL"]
ALPHA          = float(_P["ALPHA"])
DELTA          = _P["DELTA"]

LOGIT_PARAMS = {
    k: (np.inf if isinstance(v, str) and v.lower() == "inf" else v)
    for k, v in _P["LOGIT_PARAMS"].items()
}
XGB_FIXED    = dict(_P["XGB_FIXED"])
if XGB_FIXED.get("random_state") is None:
    XGB_FIXED["random_state"] = SEED
XGB_PARAMS   = dict(_P["XGB_PARAMS"])
MONO_HP_MODE = _P["MONO_HP_MODE"]

TUNE = dict(_P["TUNE"])
if TUNE.get("sampler_seed") is None:
    TUNE["sampler_seed"] = SEED
TUNE["space"] = {k: tuple(v) for k, v in TUNE["space"].items()}

EXPLAIN     = dict(_P["EXPLAIN"])
CI_METHOD   = _P["CI_METHOD"]
AUDIT_CASES = dict(_P["AUDIT_CASES"])
if AUDIT_CASES.get("seed") is None:
    AUDIT_CASES["seed"] = SEED

ROBUST = dict(_P["ROBUST"])
for _rk in ("weights", "returns", "rate_band"):
    if _rk in ROBUST and isinstance(ROBUST[_rk], list):
        ROBUST[_rk] = tuple(ROBUST[_rk])

if STAGE == "final":
    assert not PENDING, f"미결 항목이 남아 있음: {sorted(PENDING)}"

In [ ]:
# 3. 기록·해시·사전 고정 검증 함수
_CONFIG_KEYS = ["CONFIG_VERSION", "STAGE", "SEED", "SAMPLE", "PREP", "B", "B_TEST", "MIN_RATE",
                "SR_MODE", "REF_MODEL", "ALPHA", "DELTA", "LOGIT_PARAMS", "XGB_FIXED", "XGB_PARAMS",
                "MONO_HP_MODE", "TUNE", "EXPLAIN", "SIGN_TABLE", "RECOVERY_LAG", "SENS_SETS",
                "CI_METHOD", "AUDIT_CASES", "ROBUST"]

# 범위별 해시 정의: 각 단계가 실제로 의존하는 키만 묶어 무관한 변경의 과도한 차단과 STAGE/DELTA 교착을 막는다.
_SHARED_KEYS = ["SEED", "SAMPLE", "PREP", "B", "MIN_RATE", "SR_MODE", "RATES"]
_CONFIG_SCOPES = {
    "prep":     ["SAMPLE", "PREP"],
    "shared":   list(_SHARED_KEYS),
    "logit":    [*_SHARED_KEYS, "LOGIT_PARAMS"],
    "xgb":      [*_SHARED_KEYS, "XGB_FIXED", "XGB_PARAMS"],
    "xgb_mono": [*_SHARED_KEYS, "XGB_FIXED", "XGB_PARAMS", "MONO_HP_MODE"],
    "tune":     ["SEED", "SAMPLE", "PREP", "XGB_FIXED", "TUNE"],
    "select":   [*_SHARED_KEYS, "REF_MODEL", "ALPHA", "DELTA"],
    "test":     [*_SHARED_KEYS, "REF_MODEL", "ALPHA", "DELTA", "B_TEST", "EXPLAIN", "CI_METHOD", "AUDIT_CASES", "ROBUST"],
    "all":      ["SEED", "SAMPLE", "PREP", "B", "B_TEST", "MIN_RATE", "SR_MODE", "RATES",
                 "REF_MODEL", "ALPHA", "DELTA", "LOGIT_PARAMS", "XGB_FIXED", "XGB_PARAMS",
                 "MONO_HP_MODE", "TUNE", "EXPLAIN", "CI_METHOD", "AUDIT_CASES", "ROBUST"],
}

def _json_safe(v):
    """RFC-8259 표준 JSON 직렬화용 변환(np.inf → 'inf', np.nan → None, numpy 스칼라/튜플 정리)."""
    if isinstance(v, (float, np.floating)):
        f = float(v)
        if np.isnan(f):
            return None
        return str(f) if not np.isfinite(f) else f
    if isinstance(v, np.generic):
        return v.item()
    if isinstance(v, dict):
        return {str(k): _json_safe(val) for k, val in v.items()}
    if isinstance(v, (list, tuple)):
        return [_json_safe(x) for x in v]
    return v

def config_dict(keys=None):
    target = list(_CONFIG_KEYS) + ["RATES"] if keys is None else list(keys)
    d = {}
    for k in target:
        if k == "RATES":
            d["RATES"] = [float(RATES[0]), float(RATES[-1]), int(len(RATES))]
        else:
            d[k] = _json_safe(globals()[k])
    return d

def config_fingerprint(scope="shared"):
    """범위별 설정 해시(12자). 기본 scope='shared'는 ①②③ OOB 공통 비교 키만 해시한다."""
    assert scope in _CONFIG_SCOPES, f"알 수 없는 scope: {scope!r} (가능: {sorted(_CONFIG_SCOPES)})"
    s = json.dumps(config_dict(_CONFIG_SCOPES[scope]), sort_keys=True, ensure_ascii=False, allow_nan=False)
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:12]

def config_fingerprints():
    """모든 범위별 설정 해시 딕셔너리."""
    return {scope: config_fingerprint(scope) for scope in _CONFIG_SCOPES}

def sha256_file(path, chunk_size=1 << 20):
    """파일의 전체 sha256(64자 hex)을 스트리밍으로 계산한다."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            buf = f.read(chunk_size)
            if not buf:
                break
            h.update(buf)
    return h.hexdigest()

def notebook_code_sha256(nb_path):
    """노트북 코드 셀만 이어 붙여 전체 sha256(64자 hex) 계산(출력 저장으로 인한 거짓 불일치 방지)."""
    cells = json.loads(Path(nb_path).read_text(encoding="utf-8"))["cells"]
    src = "\n".join("".join(c["source"]) for c in cells if c["cell_type"] == "code")
    return hashlib.sha256(src.encode("utf-8")).hexdigest()

def notebook_code_hash(nb_path):
    """노트북 코드 셀 해시 앞 12자(lib_prep._lib_hash와 같은 규칙)."""
    return notebook_code_sha256(nb_path)[:12]

def git_head_info():
    """현재 저장소의 git HEAD·브랜치·작업트리 변경 여부를 읽기 전용으로 수집한다."""
    def _git(*args):
        try:
            r = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True)
            return r.stdout.strip()
        except Exception:
            return None
    status = _git("status", "--porcelain")
    return dict(
        head=_git("rev-parse", "HEAD"),
        branch=_git("branch", "--show-current"),
        dirty=(bool(status) if status is not None else None),
        in_colab=("google.colab" in sys.modules),
        colab_setup=globals().get("COLAB_SETUP_INFO"),
    )

def contract_hashes():
    """contract/ 핵심 약속 파일(범주·열·부호)의 sha256."""
    names = ("categories.json", "columns_logit.json", "columns_xgb.json", "signs.json")
    return {n: sha256_file(CONTRACT_DIR / n) for n in names if (CONTRACT_DIR / n).exists()}

def models_b_manifest(model_dir):
    """outputs/<model>/models_b/model_b*.joblib 전체의 정렬된 sha256 맵(모형 노트북·99_freeze 공용)."""
    mdir = Path(model_dir) / "models_b"
    if not mdir.is_dir():
        return {}
    return {f"models_b/{p.name}": sha256_file(p) for p in sorted(mdir.glob("model_b*.joblib"))}

def code_hashes():
    """공용 설정·라이브러리 코드 해시(실행 기록용)."""
    out = {}
    for name in ("00_config.ipynb", "lib_prep.ipynb", "lib_engine.ipynb", "lib_eval.ipynb", "lib_eval_add.ipynb"):
        p = NB_DIR / name
        if p.exists():
            out[name] = notebook_code_hash(p)
    cs = ROOT / "colab_setup.py"
    if cs.exists():
        out["colab_setup.py"] = sha256_file(cs)[:12]
    return out

def run_meta():
    """summary.json·FREEZE_RECORD.json 등에 넣을 공통 메타."""
    def ver(p):
        try: return metadata.version(p)
        except metadata.PackageNotFoundError: return None
    return dict(
        config=config_dict(),
        fingerprint=config_fingerprint("shared"),
        fingerprints=config_fingerprints(),
        pending=sorted(PENDING),
        lib_prep=globals().get("LIB_PREP_HASH"),
        code_hashes=code_hashes(),
        contract_hashes=contract_hashes(),
        train_csv=TRAIN_CSV.name,
        git=git_head_info(),
        python=platform.python_version(),
        packages={p: ver(p) for p in ("numpy", "pandas", "scikit-learn", "xgboost", "shap", "pyarrow")},
    )

def find_latest_freeze(require_final=True):
    """사전 고정 폴더를 찾는다(LC_FREEZE_DIR → freeze/LATEST_FREEZE.txt → 최신 freeze_*)."""
    env_dir = os.environ.get("LC_FREEZE_DIR")
    if env_dir:
        p = Path(env_dir).expanduser().resolve()
        assert (p / "FREEZE_RECORD.json").exists(), f"LC_FREEZE_DIR에 FREEZE_RECORD.json 없음: {p}"
        return p
    latest_ptr = FREEZE_DIR / "LATEST_FREEZE.txt"
    if latest_ptr.exists():
        name = latest_ptr.read_text(encoding="utf-8").strip()
        cand = (FREEZE_DIR / name) if not Path(name).is_absolute() else Path(name)
        if (cand / "FREEZE_RECORD.json").exists():
            if not require_final or not cand.name.startswith("_dryrun_"):
                return cand
    pat = "freeze_*" if require_final else "*"
    cands = sorted(p for p in FREEZE_DIR.glob(pat) if p.is_dir() and (p / "FREEZE_RECORD.json").exists())
    if not cands:
        raise FileNotFoundError(
            f"사전 고정 기록({FREEZE_DIR}/freeze_*/FREEZE_RECORD.json)이 없습니다. 먼저 99_freeze.ipynb를 DRY_RUN=False로 실행하세요."
        )
    return cands[-1]

def verify_freeze(freeze_dir=None, require_final=True, check_notebooks=True):
    """Test를 열기 전 사전 고정본(FREEZE_RECORD.json)과 현재 코드·계약·데이터·모형·선택 기록의 일치를 검증한다.
    하나라도 불일치·누락이면 AssertionError/FileNotFoundError로 즉시 중단한다."""
    fdir = find_latest_freeze(require_final=require_final) if freeze_dir is None else Path(freeze_dir).resolve()
    rec_path = fdir / "FREEZE_RECORD.json"
    assert rec_path.exists(), f"FREEZE_RECORD.json 없음: {rec_path}"
    rec = json.loads(rec_path.read_text(encoding="utf-8"))

    if require_final:
        assert STAGE == "final" and not PENDING, f"00_config가 final 상태가 아님: STAGE={STAGE}, PENDING={sorted(PENDING)}"
        assert rec.get("dry_run") is False, f"시험 고정본(_dryrun)은 본 Test에 쓸 수 없음: {fdir.name}"
        assert rec.get("stage") == "final" and not rec.get("pending"), f"고정본이 final 상태가 아님: stage={rec.get('stage')}, pending={rec.get('pending')}"

    # 1) 설정·lib_prep 해시 일치
    assert rec.get("config_hash") == config_fingerprint("shared"), (
        f"공유 config 해시 불일치: freeze={rec.get('config_hash')} vs now={config_fingerprint('shared')}"
    )
    rec_fps = rec.get("fingerprints") or {}
    if "all" in rec_fps:
        assert rec_fps["all"] == config_fingerprint("all"), (
            f"전체 config 해시 불일치: freeze={rec_fps['all']} vs now={config_fingerprint('all')}"
        )
    if "LIB_PREP_HASH" in globals():
        assert rec.get("lib_prep_hash") == LIB_PREP_HASH, (
            f"lib_prep 해시 불일치: freeze={rec.get('lib_prep_hash')} vs now={LIB_PREP_HASH}"
        )

    # 2) 노트북 코드 셀 해시(출력 저장은 허용, 코드 변경은 차단)
    if check_notebooks:
        code_map = rec.get("code_sha256") or {}
        assert code_map, "FREEZE_RECORD.json에 code_sha256이 없음(구버전 99_freeze)"
        for rel, expected in code_map.items():
            live = ROOT / rel
            assert live.exists(), f"고정된 노트북이 사라짐: {rel}"
            assert notebook_code_sha256(live) == expected, f"사전 고정 후 노트북 코드가 바뀜: {rel}"
            snap = fdir / rel
            if snap.exists():
                assert notebook_code_sha256(snap) == expected, f"고정 스냅샷 손상: {snap}"

    # 3) 계약·데이터·모형·선택 파일 원바이트 sha256 일치
    sha_map = rec.get("sha256") or {}
    assert sha_map, "FREEZE_RECORD.json에 sha256 목록이 없음"
    for rel, expected in sha_map.items():
        if rel.endswith(".ipynb"):
            continue
        live = ROOT / rel
        assert live.exists(), f"고정된 파일이 없음: {rel}"
        got = sha256_file(live)
        assert got == expected, f"사전 고정 후 파일이 바뀜(혼입 의심): {rel}"
        snap = fdir / rel
        if snap.exists():
            assert sha256_file(snap) == expected, f"고정 스냅샷 손상: {snap}"

    # 4) 고정 후 models_b에 추가된 잔여 파일 및 필수 산출물 해시 존재 여부 확인
    if require_final:
        for dev_n in ("dev_X_raw.parquet", "dev_meta.parquet"):
            assert f"data/{dev_n}" in sha_map, f"고정본에 data/{dev_n} 해시 없음"
        for m in ("logit", "xgb", "xgb_mono"):
            frozen_b = {k for k in sha_map if k.startswith(f"outputs/{m}/models_b/model_b") and k.endswith(".joblib")}
            assert len(frozen_b) == B, f"고정본에 {m} models_b 해시가 B={B}개 없음({len(frozen_b)}개)"
            live_b = {f"outputs/{m}/models_b/{p.name}" for p in (OUT_DIR / m / "models_b").glob("model_b*.joblib")}
            assert live_b == frozen_b, f"{m} models_b 파일 목록이 고정본과 다름(추가/누락)"
            for req_fn in ("summary.json", "bootstrap_results.csv", "bootstrap_curves.csv", "final_model.joblib"):
                assert f"outputs/{m}/{req_fn}" in sha_map, f"고정본에 {m}/{req_fn} 해시 없음"

    # 5) 선택 기록(selection.json) 검증 (고정본에 포함된 경우 유효성·a_star[ref_model]·입력 지문 일치 검증)
    sel = None
    sel_rel = "outputs/collect/selection.json"
    if sel_rel in sha_map:
        snap_sel = fdir / sel_rel
        assert snap_sel.exists(), f"고정 스냅샷에 selection.json 사본이 없음: {snap_sel}"
        sel = json.loads(snap_sel.read_text(encoding="utf-8"))
        assert sel.get("valid") is True, f"고정된 selection.json이 무효: {sel.get('reason')}"
        ref_m = sel.get("ref_model", REF_MODEL)
        a_map = sel.get("a_star") if isinstance(sel.get("a_star"), dict) else {}
        a_val = a_map.get(ref_m)
        assert isinstance(a_val, (int, float)) and np.isfinite(a_val) and MIN_RATE <= float(a_val) <= 1.0, (
            f"selection.json a_star[{ref_m!r}] 누락/범위 오류: {a_val!r}"
        )
        sel_inputs = sel.get("inputs") or {}
        req_sel_in = {f"{m}/{fn}" for m in ("logit", "xgb", "xgb_mono") for fn in ("summary.json", "bootstrap_results.csv", "bootstrap_curves.csv", "final_model.joblib")}
        assert req_sel_in.issubset(sel_inputs.keys()), f"selection.json inputs 필수 항목 누락: {sorted(req_sel_in - set(sel_inputs))}"
        for inp_rel, inp_h in sel_inputs.items():
            full_rel = f"outputs/{inp_rel}"
            assert sha_map.get(full_rel) == inp_h, f"selection.json 입력과 고정본 불일치: {full_rel}"

    return dict(
        freeze_dir=fdir,
        freeze_record_sha256=sha256_file(rec_path),
        record=rec,
        selection=sel,
    )

def ensure_dirs():
    for d in (DATA_DIR, CONTRACT_DIR, OUT_DIR, REPORT_DIR):
        d.mkdir(parents=True, exist_ok=True)

ensure_dirs()
_fps = config_fingerprints()
print(f"[config {CONFIG_VERSION}] 공유해시 {_fps['shared']} | 전체해시 {_fps['all']} (logit {_fps['logit']} | xgb {_fps['xgb']} | test {_fps['test']}) | STAGE={STAGE} | "
      + (f"미결 {len(PENDING)}개 ({', '.join(PENDING)}), 임시값으로 실행 중" if PENDING else "미결 없음"))

In [ ]:
# 4. 점검 (이 노트북을 직접 열었을 때만)
if not globals().get("SKIP_LIB_TESTS", False):
    print(f"ROOT={ROOT.name} | git={git_head_info()}")
    print("범위별 해시:", config_fingerprints())
    for k, v in PENDING.items():
        print(f"  미결 {k}: {v}")
    for p in (TREASURY_CSV, TRAIN_CSV, DATA_DIR / "dev_X_raw.parquet", DATA_DIR / "dev_meta.parquet",
              CONTRACT_DIR / "categories.json"):
        print("  OK  " if p.exists() else "  없음", p.relative_to(ROOT))
    test_in_raw = [p.name for p in RAW_DIR.glob("*") if "test" in p.name.lower()]
    if test_in_raw:
        print(f"  참고: raw/에 Test 파일 있음 ({test_in_raw}) — Dev 분석에는 {TRAIN_CSV.name}만 사용")